### Prerequisite

In [ ]:
import json
from sklearn.metrics import precision_score, recall_score, f1_score
import regex
import string
from tqdm import tqdm
import unicodedata
from nltk import word_tokenize

def score_recall(preds):
    # average 
    recalls = []
    for inst in preds:
        recalls.append(sum([any(preds_per_perspective) for preds_per_perspective in inst])/len(inst))

    return sum(recalls) / float(len(recalls))

def score_mrecall(preds):
    # average 
    mrecalls = []
    topk = len(preds[0][0])
    for inst in preds:  # compute mrecall for each instance
        if len(inst) > topk:
            mrecalls.append(int(sum([any(preds_per_perspective) for preds_per_perspective in inst])>=topk))
        else:
            mrecalls.append(int(all([any(preds_per_perspective) for preds_per_perspective in inst])))

    return sum(mrecalls) / float(len(mrecalls))

def score_precision(preds, topk):
    # average 
    precisions = []
    for inst in preds:
        assert len(inst[0]) >= topk, len(inst[0])
        # if len(inst[0]) < topk:
        #     topk = len(inst[0])
        num_perspective_containing_docs = 0
        for j in range(topk):
            contain_any_perspective = False
            for p in inst:
                if p[j]:
                    contain_any_perspective = True
                    break
            if contain_any_perspective:
                num_perspective_containing_docs += 1

        precisions.append(num_perspective_containing_docs / topk)

    return sum(precisions) / float(len(precisions))

def _normalize(text):
    return unicodedata.normalize('NFD', text)

def normalize_answer(s):
    def remove_articles(text):
        return regex.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def has_answer(answers, text, tokenizer) -> bool:
    """Check if a document contains an answer string."""
    text = _normalize(text)
    text = tokenizer.tokenize(text, uncased=True)

    for answer in answers:
        answer = _normalize(answer)
        answer = tokenizer.tokenize(answer, uncased=True)
        for i in range(0, len(text) - len(answer) + 1):
            if answer == text[i: i + len(answer)]:
                return True
    return False


class SimpleTokenizer(object):
    ALPHA_NUM = r'[\p{L}\p{N}\p{M}]+'
    NON_WS = r'[^\p{Z}\p{C}]'

    def __init__(self):
        """
        Args:
            annotators: None or empty set (only tokenizes).
        """
        self._regexp = regex.compile(
            '(%s)|(%s)' % (self.ALPHA_NUM, self.NON_WS),
            flags=regex.IGNORECASE + regex.UNICODE + regex.MULTILINE
        )

    def tokenize(self, text, uncased=False):
        matches = [m for m in self._regexp.finditer(text)]
        if uncased:
            tokens = [m.group().lower() for m in matches]
        else:
            tokens = [m.group() for m in matches]
        return tokens

def read_jsonl(file_path):
    import json
    with open(file_path, 'r') as file:
        return [json.loads(line) for line in file]
    
    
def load_data(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

### Process Answer Function

In [ ]:

def process_answer(answer_str):
    if isinstance(answer_str, list):
        return [normalize_answer(ans) for ans in answer_str]
    answer_list = []
    for ans in answer_str.split('\n'):
        ans = ' '.join(ans.split(' ')[1:])
        # answer_list.append(normalize_answer(ans.strip()[3:].strip()))
        answer_list.append(normalize_answer(ans))
    return answer_list

def process_reference(inst):
    # return a list
    answer_list = []
    for answer in inst['answer_list']:
        answer_list.append(answer['aliases'])
    answer_list = [item for sublist in answer_list for item in sublist]
    answer_list = [normalize_answer(ans) for ans in answer_list]
    return answer_list

### Given prediction & reference, compute scores

In [ ]:
def evaluate(predictions, references):
    predictions = set(list(predictions))
    references = set(list(references))
    num_intersections = 0
    for pred in predictions:
        for ref in references:
            if pred == ref:
                num_intersections += 1
                break
    if len(predictions) == 0:
        return 0, 0, 0
    precision = num_intersections / len(predictions)
    recall = num_intersections / len(references)
    if precision == 0 or recall == 0:
        f1 = 0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
    return precision, recall, f1

### Evaluating Retrieval

In [ ]:

def eval_retrieve_docs(retrieved_docs_path, data_path, has_gold_id=False, topk=100):
    dataset = read_jsonl(data_path)
    tok = SimpleTokenizer()
    retrieved_docs = read_jsonl(retrieved_docs_path)
    len_docs = []
    # precisions = []
    preds = []
    
    for gold_inst, docs in tqdm(zip(dataset, retrieved_docs)):
        pred_inst = []
        # valid_answers = process_reference(gold_inst)
        # print(valid_answers)
        
        # for ans in valid_answers:
        if has_gold_id:
            gold_indices = [doc['id'] for doc in gold_inst['positive_ctxs']]
            for gold_index in gold_indices:
                pred_inst.append([])
                len_docs.append(len(docs['ctxs']))
                for doc in docs['ctxs'][:topk]:
                    # print(gold_index, doc['id'])
                    pred = gold_index == doc['id']
                    pred_inst[-1].append(pred)
        else:
            for answer in gold_inst['answer_list']:
                pred_inst.append([])
                len_docs.append(len(docs['ctxs']))
                for doc in docs['ctxs'][:topk]:
                    pred = has_answer(answer['aliases'], doc['text'], tok)
                    pred_inst[-1].append(pred)
        preds.append(pred_inst)    
        # precisions.append(sum([any(doc[pred_str]) for doc in docs[:topk]]) / topk)
        
    mrecall_score = score_mrecall(preds)
    recall_score = score_recall(preds)
    precision_score = score_precision(preds, topk)
    print(f'Precision: {100*precision_score:.2f}')
    print(f'Recall: {100*recall_score:.2f}')
    print(f'MRecall: {100*mrecall_score:.2f}')
    print('Average number of retrieved documents:', sum(len_docs) / len(len_docs)) 
    return precision_score, mrecall_score

### Evaluating whether repeat input documents

In [ ]:

def eval_retrieve_docs_for_repeats(retrieved_docs_path, data_path, topk=100):
    dataset = read_jsonl(data_path)
    tok = SimpleTokenizer()
    retrieved_docs = read_jsonl(retrieved_docs_path)
    len_docs = []
    # precisions = []
    preds = []
    
    for gold_inst, docs in tqdm(zip(dataset, retrieved_docs)):
        pred_inst = []
        # valid_answers = process_reference(gold_inst)
        # print(valid_answers)
        
        # for ans in valid_answers:
        gold_indices = [doc['id'] for doc in gold_inst['hard_negative_ctxs']]
        for gold_index in gold_indices:
            pred_inst.append([])
            len_docs.append(len(docs['ctxs']))
            for doc in docs['ctxs'][:topk]:
                # print(gold_index, doc['id'])
                pred = gold_index == doc['id']
                pred_inst[-1].append(pred)
        
        preds.append(pred_inst)    
        # precisions.append(sum([any(doc[pred_str]) for doc in docs[:topk]]) / topk)
        
    mrecall_score = score_mrecall(preds)
    recall_score = score_recall(preds)
    precision_score = score_precision(preds, topk)
    print(f'Precision: {100*precision_score:.2f}')
    print(f'Recall: {100*recall_score:.2f}')
    print(f'MRecall: {100*mrecall_score:.2f}')
    print('Average number of retrieved documents:', sum(len_docs) / len(len_docs)) 
    return precision_score, mrecall_score

### Main Function

In [ ]:

def main():
    # Load QAMpari dev data
    # model = "gemma-2-9b-it"
    model = "Meta-Llama-3.1-8B-Instruct"
    # model = "gpt-4o-mini"
    dataset = read_jsonl('data/qampari_data/dev_data.jsonl')
    pred_data = read_jsonl(f'outputs/QAMPARI/{model}_qampari_rnone.jsonl')
    # pred_data = read_jsonl(f'output_wo_rag.jsonl')
    
    # Assuming dev_data is a list of dictionaries with 'prediction' and 'reference' keys
    assert len(dataset) == len(pred_data), (len(dataset), len(pred_data))
    predictions = []
    references = []
    
    for gold_inst, pred_inst in zip(dataset, pred_data):
        # ans_text = pred_inst['answer'] if 
        # num_w = len(word_tokenize())
        # num_words.append(len(word_tokenize()))
        predictions.append(process_answer(pred_inst['answer']))
        references.append(process_reference(gold_inst))
    
    # Evaluate
    precisions = []
    recalls = []
    f1s = []
    num_answers = []
    repeats = []
    for pred, ref in zip(predictions, references):
        num_answers.append(len(set(pred)))
        repeats.append(len(pred) - len(set(pred)))
        precision, recall, f1 = evaluate(pred, ref)
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    
    avg_precision = sum(precisions) / len(precisions)
    avg_recall = sum(recalls) / len(recalls)
    avg_f1 = sum(f1s) / len(f1s)
    avg_num_answers = sum(num_answers) / len(num_answers)
    avg_repeats = sum(repeats) / len(repeats)
    
    print(f'Average Precision: {100*avg_precision:.2f}')
    print(f'Average Recall: {100*avg_recall:.2f}')
    print(f'Average F1 Score: {100*avg_f1:.2f}')
    print(f'Average Number of Answers: {avg_num_answers:.2f}')
    print(f'Average Number of Repeats: {avg_repeats:.2f}')

# Evaluate Retrieval for Different Retrievers

### Evaluating Retrieval for Standard / Reranking 

In [ ]:
# for retriever in ['bm25', 'nv', 'contriever']:
    #     print(retriever)
    #     eval_retrieve_docs(
    #         f'/scratch/cluster/hungting/projects/diverse_response/retrieval_outputs/qampari/{retriever}_reranked_l0.9.jsonl', 
    #         'data/qampari_data/dev_data.jsonl',
    #         topk=5
    #     )
    # for _lambda in [0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99]:
    #     print('lambda:', _lambda)   
    #     eval_retrieve_docs(
    #             f'/scratch/cluster/hungting/projects/diverse_response/retrieval_outputs/qampari/reranked/bm25_reranked_l{_lambda}.jsonl', 
    #             'data/qampari_data/dev_data.jsonl',
    #             topk=5
    #         )

### Evaluating Subtraction Results

In [ ]:
# eval_retrieve_docs(
    #     f'/scratch/cluster/hungting/projects/Multi_Answer/mteb_retriever/outputs/nv-embed/qampari_dev_autoregressive.jsonl', 
    #     'data/qampari_data/dev_data.jsonl',
    #     topk=5
    # )
    # eval_retrieve_docs(
    #     f'/scratch/cluster/hungting/projects/Multi_Answer/contriever/outputs/contriever_msmarco_wiki/qampari_dev_autoregressive.jsonl', 
    #     'data/qampari_data/dev_data.jsonl',
    #     topk=5
    # )

### Evaluating Second Stage Results

In [ ]:
eval_retrieve_docs(
    '/scratch/cluster/hungting/projects/Multi_Answer/contriever/outputs/contriever_msmarco_wiki_2nd_stage/qampari_dev_2nd_stage_org.json',
    '../../../data/qampari_data/2nd_stage/dev_data.jsonl',
    has_gold_id=True,
    topk=5
)

eval_retrieve_docs_for_repeats(
    '/scratch/cluster/hungting/projects/Multi_Answer/contriever/outputs/contriever_msmarco_wiki_2nd_stage/qampari_dev_2nd_stage_org.json',
    '../../../data/qampari_data/2nd_stage/dev_data.jsonl',
    topk=5
)